In [ ]:
import torch
import math
import matplotlib.pyplot as plt
import os
import sys

sys.path.append('.')
from methods.pinn import create_heat_pinn_factory
from methods.fno import create_fno_factory
from harness.measurement import ReconciledMeasurementHarness

def run_frequency_stress_test(pinn_factory, fno_factory, test_ks):
    print("\n=== STAGE 2: HIGH-FREQUENCY STRESS TEST ===")
    x_test = torch.linspace(0, 1, 100).view(-1, 1)
    
    fig, axes = plt.subplots(1, len(test_ks), figsize=(12, 5))
    if len(test_ks) == 1:
        axes = [axes]

    for idx, k in enumerate(test_ks):
        print(f"--- Evaluating Wave Frequency: k = {k} ---")
        decay = math.exp(-0.01 * (k * math.pi)**2 * 0.01)
        exact_u = torch.sin(k * math.pi * x_test) * decay
        
        u_pinn, _ = pinn_factory(k)
        u_fno, _ = fno_factory(k)
        
        err_pinn = ReconciledMeasurementHarness.relative_l2_error(u_pinn, exact_u)
        err_fno = ReconciledMeasurementHarness.relative_l2_error(u_fno, exact_u)
        
        ax = axes[idx]
        ax.plot(x_test.numpy(), exact_u.numpy(), 'k-', linewidth=2, label='Exact Solution')
        ax.plot(x_test.numpy(), u_pinn.detach().numpy(), 'r--', linewidth=2, label=f'PINN (Err: {err_pinn:.2f})')
        ax.plot(x_test.numpy(), u_fno.detach().numpy(), 'b:', linewidth=2, label=f'FNO (Err: {err_fno:.2f})')
        ax.set_title(f"Frequency: k = {k}", fontsize=12)
        ax.set_xlabel("x", fontsize=10)
        if idx == 0:
            ax.set_ylabel("u(x, t=0.01)", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/frequency_stress_test.png', dpi=300)
    print("Saved plot to 'figures/frequency_stress_test.png'.")

if __name__ == "__main__":
    torch.manual_seed(42)
    test_pinn_factory = create_heat_pinn_factory(alpha=0.01, epochs=3000)
    test_fno_factory = create_fno_factory(epochs=100, num_samples=2000)
    run_frequency_stress_test(test_pinn_factory, test_fno_factory, test_ks=[4, 25])